In [ ]:
import altair as alt
import gcsfs
import pandas as pd

import _operator_report_utils as utils
from update_vars import (
    DIGEST_DICT, PROCESSED_GCS, 
    abbrev_month, readable_dict, analysis_month
)

alt.data_transformers.enable("vegafusion")

In [ ]:
import _portfolio_charts

In [ ]:
analysis_name = "Alameda-Contra Costa Transit District"

schedule_rt_route_direction_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.schedule_rt_route_direction}_{abbrev_month}.parquet"

df = pd.read_parquet(
    schedule_rt_route_direction_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[[("Analysis Name", "==", analysis_name)]]
).reset_index(drop=True)

In [ ]:
# was in create_route_dropdown
routes_list = df.Route.unique().tolist()

route_dropdown = alt.binding_select(
    options=routes_list,
    name="Routes: ",
)

# Column that controls the bar charts
xcol_param = alt.selection_point(
    fields=["Route"], value=routes_list[0], bind=route_dropdown
)

In [ ]:
# Set drop down menu to be on the upper right for the charts
from IPython.display import HTML
display(
    HTML(
        """
<style>
form.vega-bindings {
  position: absolute;
  right: 0px;
  top: 0px;
}
</style>
"""
    )
)

In [ ]:
day_type_order = ["Weekday", "Saturday", "Sunday"]
WIDTH = 200 * 2
HEIGHT = 250

chart_scheduled_minutes = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x="Date",
        y="Average Scheduled Minutes",
        color=alt.Color("Day Type:N", scale=alt.Scale(domain=day_type_order)),
        column="Direction:O",
        tooltip = ["Date", "Average Scheduled Minutes", "Route", "Direction", "Day Type"]
    ).transform_filter(xcol_param).properties(width=WIDTH, height=HEIGHT)
)

In [ ]:
chart_frequency = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x="Date",
        y="Headway All Day",
        color=alt.Color("Day Type:N", scale=alt.Scale(domain=day_type_order)),
        column="Direction:O"
    ).transform_filter(xcol_param).properties(width=WIDTH, height=HEIGHT)
)

In [ ]:
chart_peak = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x="Date",
        y="Headway Peak",
        color=alt.Color("Day Type:N", scale=alt.Scale(domain=day_type_order)),
        column="Direction:O"
    ).transform_filter(xcol_param).properties(width=WIDTH, height=HEIGHT)
)

In [ ]:
combined_chart = alt.vconcat(chart_scheduled_minutes, chart_frequency, chart_peak).add_params(xcol_param)
combined_chart

In [ ]:
dir_0_chart = _portfolio_charts.bar_chart(
        df=df2.loc[df2.Direction == 0],
        x_col="Date",
        y_col="Average Scheduled Minutes",
        color_col="Direction",
        color_scheme=[*chart_dict.colors],
        tooltip_cols=list(chart_dict.tooltip),
        date_format="",
        y_ticks=chart_dict.ticks,
    )


In [ ]:
def create_scheduled_minutes(df: pd.DataFrame):
    df2 = df.loc[df["Day Type"] == "Weekday"]
    chart_dict = readable_dict.avg_scheduled_minutes

    xcol_param = create_route_dropdown(df)

    dir_0_chart = _portfolio_charts.bar_chart(
        df=df2.loc[df2.Direction == 0],
        x_col="Date",
        y_col="Average Scheduled Minutes",
        color_col="Direction",
        color_scheme=[*chart_dict.colors],
        tooltip_cols=list(chart_dict.tooltip),
        date_format="",
        y_ticks=chart_dict.ticks,
    )

    dir_0_chart = (
        _portfolio_charts.configure_chart(
            dir_0_chart,
            width=200,
            height=250,
            title=f"{chart_dict.title} for Direction 0",
            subtitle=chart_dict.subtitle,
        )
        .add_params(xcol_param)
        .transform_filter(xcol_param)
    )

    dir_1_chart = _portfolio_charts.bar_chart(
        df=df2.loc[df2.Direction == 1],
        x_col="Date",
        y_col="Average Scheduled Minutes",
        color_col="Direction",
        color_scheme=[*chart_dict.colors],
        tooltip_cols=list(chart_dict.tooltip),
        date_format="",
        y_ticks=chart_dict.ticks,
    )
    dir_1_chart = (
        _portfolio_charts.configure_chart(
            dir_1_chart,
            width=200,
            height=250,
            title="Direction 1",
            subtitle="",
        )
        .add_params(xcol_param)
        .transform_filter(xcol_param)
    )
    chart = alt.hconcat(dir_0_chart, dir_1_chart)
    return chart


In [ ]:
try:
    display(utils.create_scheduled_minutes(schedule_rt_route_direction_summary_df))
    display(utils.create_frequency(schedule_rt_route_direction_summary_df))
    display(utils.create_text_graph(schedule_rt_route_direction_summary_df))
except:
    display(Markdown(f"""{analysis_name} doesn't have detailed route information."""))